# End-to-End TTS Training (Fully Updated)
This notebook has all the path fixes, library installations, and explicit Google Drive mounting built directly into it.

In [ ]:
# Cell 1: Mount Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install g2p_en soundfile librosa pandas tqdm --quiet

import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('cmudict', quiet=True)
print("✅ Dependencies installed and Drive mounted!")

In [ ]:
# Cell 2: Clone Repository
import os
import sys

%cd /content
if not os.path.exists('/content/ML-miniproject'):
    !git clone https://github.com/Roshan2607/ML-miniproject.git /content/ML-miniproject

%cd /content/ML-miniproject
sys.path.append('/content/ML-miniproject')
print("✅ Repository cloned and path set!")

In [ ]:
# Cell 3: Download LJSpeech
import os
import glob

if not os.path.exists('LJSpeech-1.1'):
    !wget https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
    !tar -xjf LJSpeech-1.1.tar.bz2

wav_files = glob.glob('LJSpeech-1.1/wavs/*.wav')
print(f"✅ Found {len(wav_files)} wav files.")

In [ ]:
# Cell 4: Preprocess in batches
import os
import numpy as np
from tqdm import tqdm
from src.preprocessing import audio_to_spectrogram
from src.config import Config

os.makedirs(Config.PROCESSED_DATA_DIR, exist_ok=True)
wav_files = glob.glob('LJSpeech-1.1/wavs/*.wav')

BATCH_SIZE = 500
print("Starting preprocessing...")
for i in tqdm(range(0, len(wav_files), BATCH_SIZE)):
    batch = wav_files[i:i+BATCH_SIZE]
    for wav_file in batch:
        base_name = os.path.basename(wav_file).replace('.wav', '.npy')
        out_path = os.path.join(Config.PROCESSED_DATA_DIR, base_name)
        if not os.path.exists(out_path):
            spec = audio_to_spectrogram(wav_file)
            np.save(out_path, spec)
print("✅ Preprocessing complete!")

In [ ]:
# Cell 5: Training Configuration
MAX_SAMPLES = 2000
BATCH_SIZE = 16
NUM_EPOCHS = 25
LEARNING_RATE = 0.001
TEACHER_FORCING_RATIO = 0.5
TRAIN_SPLIT = 0.9

print("✅ Configuration set.")

In [ ]:
# Cell 6: Create DataLoaders
import torch
from torch.utils.data import DataLoader, random_split
from src.dataset import LJSpeechDataset, collate_fn
from src.config import Config

# Using the absolute path inside the project folder
metadata_path = '/content/ML-miniproject/LJSpeech-1.1/metadata.csv'

dataset = LJSpeechDataset(metadata_path=metadata_path, processed_dir=Config.PROCESSED_DATA_DIR)

if len(dataset) > MAX_SAMPLES:
    indices = list(range(MAX_SAMPLES))
    dataset = torch.utils.data.Subset(dataset, indices)

train_size = int(TRAIN_SPLIT * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"✅ Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

In [ ]:
# Cell 7: Initialize Model
import torch
from src.model import Seq2SeqTTS
from src.config import Config
from src.preprocessing import get_vocab_size

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vocab_size = get_vocab_size()

model = Seq2SeqTTS(vocab_size, Config.embed_dim, Config.hidden_dim, Config.spec_bins).to(device)

param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model initialized on {device}. Parameters: {param_count:,}")

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# Cell 8: Training Loop
import os
import json
import torch

def masked_mse_loss(prediction, target, lengths):
    batch_size, max_time, _ = target.shape
    time_indices = torch.arange(max_time, device=target.device).unsqueeze(0)
    mask = (time_indices < lengths.unsqueeze(1))
    frame_loss = torch.mean((prediction - target) ** 2, dim=2)
    frame_loss = frame_loss * mask.float()
    denominator = mask.float().sum().clamp(min=1.0)
    return frame_loss.sum() / denominator

CHECKPOINT_DIR = '/content/drive/MyDrive/tts_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_train_loss = 0.0
    
    for text, seq_lengths, target_spec, spec_lengths in train_loader:
        text, seq_lengths = text.to(device), seq_lengths.to(device)
        target_spec, spec_lengths = target_spec.to(device), spec_lengths.to(device)
        
        optimizer.zero_grad()
        prediction = model(text, seq_lengths, target_spec, TEACHER_FORCING_RATIO)
        loss = masked_mse_loss(prediction, target_spec, spec_lengths)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_train_loss += loss.item()
        
    epoch_train_loss /= len(train_loader)
    train_losses.append(epoch_train_loss)
    
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for text, seq_lengths, target_spec, spec_lengths in val_loader:
            text, seq_lengths = text.to(device), seq_lengths.to(device)
            target_spec, spec_lengths = target_spec.to(device), spec_lengths.to(device)
            
            prediction = model(text, seq_lengths, target_spec, teacher_forcing_ratio=0.0)
            loss = masked_mse_loss(prediction, target_spec, spec_lengths)
            epoch_val_loss += loss.item()
            
    epoch_val_loss /= len(val_loader)
    val_losses.append(epoch_val_loss)
    
    print(f"Epoch {epoch}/{NUM_EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
    
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val_loss
        }, os.path.join(CHECKPOINT_DIR, 'best_model.pt'))
        print("Saved new best model.")
        
with open(os.path.join(CHECKPOINT_DIR, 'loss_history.json'), 'w') as f:
    json.dump({'train_loss': train_losses, 'val_loss': val_losses}, f)

In [ ]:
# Cell 9: Plot Training Curves
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(range(1, NUM_EPOCHS + 1), train_losses, label='Train Loss')
plt.plot(range(1, NUM_EPOCHS + 1), val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Masked MSE Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Cell 10: Inference & Listen
import torch
from src.preprocessing import text_to_sequence, spectrogram_to_audio
import soundfile as sf
import IPython.display as ipd

# Load best model
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best_model.pt'), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

text = "Welcome to our machine learning mini project."
print(f"Generating audio for: '{text}'")

with torch.no_grad():
    seq = torch.LongTensor(text_to_sequence(text)).unsqueeze(0).to(device)
    seq_lengths = torch.LongTensor([seq.size(1)]).to(device)
    
    # Dummy target to define output length (approx 3-4 seconds)
    dummy_target = torch.zeros(1, 300, Config.spec_bins).to(device)
    
    predicted_spec = model(seq, seq_lengths, dummy_target, teacher_forcing_ratio=0.0)
    
spec_np = predicted_spec.squeeze(0).cpu().numpy()
audio = spectrogram_to_audio(spec_np)

sf.write('output.wav', audio, Config.sample_rate)
ipd.Audio('output.wav')